# Predictive Maintenance Analysis

## Objective
Analyze machine operating conditions and identify factors that contribute to machine failures, in order to support predictive maintenance decisions.

## Business Problem
Unexpected machine failures increase downtime, maintenance costs, and productivity losses. This analysis aims to identify patterns that can help reduce failures through preventive maintenance.

In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', None)

# Consistent blue color palette used across all charts
BLUE_DARK = '#0B3D66'
BLUE_MAIN = '#1668A5'
BLUE_MED = '#3A8DCB'
BLUE_LIGHT = '#7FB8E0'
BLUE_PALE = '#CBE3F5'
BLUE_SEQUENCE = [BLUE_PALE, BLUE_LIGHT, BLUE_MED, BLUE_MAIN, BLUE_DARK]

pio.templates.default = 'plotly_white'

## 1. Data Loading & Quality Check

In [2]:
df = pd.read_csv('/content/Predictive_Maintenance_Cleaned.csv')

df.head()

,Type,Air_Temperature,Process_Temperature,Rotational_Speed,Torque,Tool_Wear,Machine_Failure,Tool_Wear_Failure,Heat_Dissipation_Failure,Power_Failure,Overstrain_Failure,Random_Failure
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [3]:
print(df.shape)

(10000, 12)


In [4]:
df.isnull().sum()

,0
Type,0
Air_Temperature,0
Process_Temperature,0
Rotational_Speed,0
Torque,0
Tool_Wear,0
Machine_Failure,0
Tool_Wear_Failure,0
Heat_Dissipation_Failure,0
Power_Failure,0


In [5]:
df.duplicated().sum()

np.int64(0)

## 2. Key Business Questions

### What is the total number of operations?

In [6]:
len(df)

10000

### What is the machine failure rate?

In [7]:
failure_rate = round(df['Machine_Failure'].mean() * 100, 2)
non_failure_rate = round(100 - failure_rate, 2)

print(f"Machine Failure Rate: {failure_rate}%")
print(f"Non-Failure Rate: {non_failure_rate}%")

Machine Failure Rate: 3.39%
Non-Failure Rate: 96.61%


In [8]:
df['Failure_Label'] = df['Machine_Failure'].map({0: 'No Failure', 1: 'Failure'})

failure_dist = df['Failure_Label'].value_counts().reindex(['No Failure', 'Failure'])
failure_dist_pct = (failure_dist / failure_dist.sum() * 100).round(1)

fig = go.Figure(go.Bar(
    x=failure_dist.index,
    y=failure_dist.values,
    marker_color=[BLUE_LIGHT, BLUE_DARK],
    text=[f'{p}%' for p in failure_dist_pct.values],
    textposition='outside'
))

fig.update_layout(
    title='Machine Failure Distribution',
    xaxis_title='',
    yaxis_title='Count',
    yaxis=dict(range=[0, failure_dist.values.max() * 1.2]),
    height=420,
    margin=dict(t=80)
)
fig.show()

Only 3% of machines experienced failures, indicating that failures are relatively rare events.

### What is the distribution of product types?

In [9]:
type_counts = df['Type'].value_counts()

fig = go.Figure(go.Bar(
    x=type_counts.index,
    y=type_counts.values,
    marker_color=BLUE_SEQUENCE[1:4],
    text=type_counts.values,
    textposition='outside'
))

fig.update_layout(
    title='Product Type Distribution',
    xaxis_title='Product Type',
    yaxis_title='Count',
    yaxis=dict(range=[0, type_counts.values.max() * 1.2]),
    height=420,
    margin=dict(t=80)
)
fig.show()

Type L machines represent the largest portion of the dataset.

### Which product type experiences the highest failure rate?

In [10]:
failure_rate_type = (
    (df.groupby('Type')['Machine_Failure'].mean() * 100)
    .round(2)
    .sort_values(ascending=False)
)

failure_rate_type

,Machine_Failure
Type,
L,3.92
M,2.77
H,2.09


In [11]:
fig = go.Figure(go.Bar(
    x=failure_rate_type.index,
    y=failure_rate_type.values,
    marker_color=[BLUE_DARK, BLUE_MED, BLUE_LIGHT],
    text=[f'{v:.2f}%' for v in failure_rate_type.values],
    textposition='outside'
))

fig.update_layout(
    title='Failure Rate by Product Type',
    xaxis_title='Product Type',
    yaxis_title='Failure Rate (%)',
    yaxis=dict(range=[0, failure_rate_type.values.max() * 1.2]),
    height=420,
    margin=dict(t=80)
)
fig.show()

Type L machines exhibit the highest failure rate and may require more frequent maintenance.

### Does tool wear increase machine failure risk?

In [12]:
fig = px.violin(
    df,
    x='Failure_Label',
    y='Tool_Wear',
    color='Failure_Label',
    color_discrete_map={'No Failure': BLUE_LIGHT, 'Failure': BLUE_DARK},
    box=True
)

fig.update_layout(
    title='Tool Wear Distribution by Failure Status',
    xaxis_title='',
    yaxis_title='Tool Wear',
    showlegend=False,
    height=450
)
fig.show()

Failed machines generally show significantly higher tool wear values.

### How does failure rate change across tool wear ranges?

In [13]:
df['Wear_Group'] = pd.cut(
    df['Tool_Wear'],
    bins=[0, 50, 100, 150, 200, 250],
    labels=['0-50', '50-100', '100-150', '150-200', '200-250']
)

In [14]:
wear_failure = (
    df.groupby('Wear_Group', observed=False)['Machine_Failure']
      .mean()
      .mul(100)
      .round(2)
)

wear_failure

,Machine_Failure
Wear_Group,
0-50,2.15
50-100,2.33
100-150,2.18
150-200,2.90
200-250,15.26


In [15]:
fig = go.Figure(go.Scatter(
    x=wear_failure.index.astype(str),
    y=wear_failure.values,
    mode='lines+markers',
    line=dict(color=BLUE_MAIN, width=3),
    marker=dict(size=9, color=BLUE_DARK),
    fill='tozeroy',
    fillcolor='rgba(22, 104, 165, 0.12)'
))

fig.update_layout(
    title='Failure Rate by Tool Wear Group',
    xaxis_title='Tool Wear Group',
    yaxis_title='Failure Rate (%)',
    height=450
)
fig.show()

Failure rates increase as tool wear increases.

### What operating conditions are most common among failed machines?

In [16]:
df.groupby('Failure_Label')[[
    'Air_Temperature',
    'Process_Temperature',
    'Rotational_Speed',
    'Torque',
    'Tool_Wear'
]].mean().round(2)

,Air_Temperature,Process_Temperature,Rotational_Speed,Torque,Tool_Wear
Failure_Label,,,,,
Failure,300.89,310.29,1496.49,50.17,143.78
No Failure,299.97,310.00,1540.26,39.63,106.69


In [17]:
cols = [
    'Air_Temperature',
    'Process_Temperature',
    'Rotational_Speed',
    'Torque',
    'Tool_Wear'
]

comparison = (
    df.groupby('Failure_Label')[cols]
      .mean()
      .round(2)
      .T
)

comparison

Failure_Label,Failure,No Failure
Air_Temperature,300.89,299.97
Process_Temperature,310.29,310.00
Rotational_Speed,1496.49,1540.26
Torque,50.17,39.63
Tool_Wear,143.78,106.69


Failed machines tend to operate with higher tool wear, higher torque, and more stressful operating conditions.

### Which failure type occurs most frequently?

In [18]:
failure_cols = [
    'Tool_Wear_Failure',
    'Heat_Dissipation_Failure',
    'Power_Failure',
    'Overstrain_Failure',
    'Random_Failure'
]

failure_counts = df[failure_cols].sum().sort_values(ascending=False)

failure_summary = pd.DataFrame({
    'Count': failure_counts,
    'Percentage (%)': (failure_counts / failure_counts.sum() * 100).round(2)
})

failure_summary

,Count,Percentage (%)
Heat_Dissipation_Failure,115,30.83
Overstrain_Failure,98,26.27
Power_Failure,95,25.47
Tool_Wear_Failure,46,12.33
Random_Failure,19,5.09


In [19]:
fig = go.Figure(go.Bar(
    x=failure_counts.index,
    y=failure_counts.values,
    marker_color=BLUE_SEQUENCE[::-1][:len(failure_counts)],
    text=failure_counts.values,
    textposition='outside'
))

fig.update_layout(
    title='Failure Types',
    xaxis_title='',
    yaxis_title='Count',
    yaxis=dict(range=[0, failure_counts.values.max() * 1.2]),
    xaxis_tickangle=-45,
    height=450,
    margin=dict(t=80)
)
fig.show()

The most common failure type contributes the largest share of machine breakdowns and should be prioritized in maintenance planning.

### Does excessive mechanical stress increase failure risk?

In [20]:
df['Mechanical_Stress'] = df['Torque'] * df['Tool_Wear']

In [26]:
fig = px.scatter(
    df,
    x='Rotational_Speed',
    y='Torque',
    color='Failure_Label',
    size='Mechanical_Stress',
    color_discrete_map={
        'No Failure': BLUE_LIGHT,
        'Failure': BLUE_DARK
    }
)

fig.update_layout(
    title=dict(
        text='<b>Mechanical Stress & Failure</b>',
        x=0.5,
        font=dict(color='#044b94')
    )
)
fig.show()

Machines exposed to high mechanical stress show significantly higher failure rates.

### Can we identify an early warning indicator before failure occurs?

In [22]:
df.groupby('Failure_Label')[[
    'Tool_Wear',
    'Torque',
    'Process_Temperature',
    'Rotational_Speed'
]].mean().round(2)

,Tool_Wear,Torque,Process_Temperature,Rotational_Speed
Failure_Label,,,,
Failure,143.78,50.17,310.29,1496.49
No Failure,106.69,39.63,310.00,1540.26


### What are the most important factors driving machine failures?

In [23]:
df_corr = df.copy()

le = LabelEncoder()
df_corr['Type'] = le.fit_transform(df_corr['Type'])

In [24]:
corr = df_corr.drop(columns=['Wear_Group', 'Failure_Label']).corr().round(2)

fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale=['#FFFFFF', BLUE_LIGHT, BLUE_MAIN, BLUE_DARK],
    zmin=-1,
    zmax=1,
    aspect='auto'
)

fig.update_layout(
    title='Correlation Heatmap',
    height=600
)
fig.show()

## Business Insights

1. **Heat-related issues are the biggest cause of machine breakdowns.**
   Machines are more likely to fail when heat is not properly managed, making thermal monitoring a key area for reducing downtime.

2. **Worn tools significantly increase the risk of failure.**
   Machines operating with highly worn tools experience much higher failure rates, indicating that timely tool replacement can prevent costly breakdowns.

3. **Machines under heavy workload are more likely to fail.**
   Higher torque and mechanical stress are commonly observed in failed machines, suggesting that excessive operating loads increase failure risk.

4. **Most failures can be prevented through proactive maintenance.**
   The analysis shows clear warning signs before failures occur, especially related to tool wear and operating conditions, creating opportunities for preventive maintenance.

## Business Recommendations

1. Replace or inspect tools before they reach high wear levels to reduce unexpected machine downtime.
2. Strengthen monitoring of machine temperature and cooling systems to minimize heat-related failures.
3. Track machine workload continuously and avoid operating under excessive stress for long periods.
4. Adopt a preventive maintenance strategy instead of waiting for failures to occur.
5. Use operational indicators such as tool wear, torque, and temperature as early warning signals to identify machines at risk.